# Import Libraries

In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import joblib
from joblib import Parallel, delayed

from collections import defaultdict
from torch.nn import GRU as GRU_setup, Linear as linear_method, Module as TorchModule
from torch.optim import Adam as model_optimizer
from torch.nn import MSELoss as loss_function
from catboost import CatBoostRegressor
from prophet import Prophet
from prophet.diagnostics import cross_validation
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error

from Utils.file_checker import FileChecker

In [2]:
DATASET_PATH = 'Source\Dataset\WB_SE4ALL.csv'
target_output = 'Output'

In [3]:
df = pd.read_csv(DATASET_PATH)
df.head(n=10)

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,SEX,...,UNIT_MULT,UNIT_MULT_LABEL,UNIT_TYPE,UNIT_TYPE_LABEL,TIME_FORMAT,TIME_FORMAT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL
0,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
1,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
2,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
3,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
4,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
5,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
6,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
7,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
8,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public
9,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,Annual,AFG,Afghanistan,WB_SE4ALL_EG_ACS_ELEC,Access to electricity (% of population),_T,...,0,Units,RATIO,Ratio,602,CCYY,A,Normal value,PU,Public


# Initial Data Analysis (IDA)

In [4]:
print(f"Data shape : {df.shape}")

Data shape : (144418, 39)


In [5]:
print(f"Data types : {df.dtypes}")

Data types : STRUCTURE                  object
STRUCTURE_ID               object
ACTION                     object
FREQ                       object
FREQ_LABEL                 object
REF_AREA                   object
REF_AREA_LABEL             object
INDICATOR                  object
INDICATOR_LABEL            object
SEX                        object
SEX_LABEL                  object
AGE                        object
AGE_LABEL                  object
URBANISATION               object
URBANISATION_LABEL         object
UNIT_MEASURE               object
UNIT_MEASURE_LABEL         object
COMP_BREAKDOWN_1           object
COMP_BREAKDOWN_1_LABEL     object
COMP_BREAKDOWN_2           object
COMP_BREAKDOWN_2_LABEL     object
COMP_BREAKDOWN_3           object
COMP_BREAKDOWN_3_LABEL     object
TIME_PERIOD                 int64
OBS_VALUE                 float64
AGG_METHOD                 object
AGG_METHOD_LABEL           object
DATABASE_ID                object
DATABASE_ID_LABEL          object
U

In [6]:
constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
print("\nConstant Columns:", constant_cols)
print(f"Total Constant: {len(constant_cols)}")


Constant Columns: ['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'FREQ', 'FREQ_LABEL', 'SEX', 'SEX_LABEL', 'AGE', 'AGE_LABEL', 'COMP_BREAKDOWN_2', 'COMP_BREAKDOWN_2_LABEL', 'COMP_BREAKDOWN_3', 'COMP_BREAKDOWN_3_LABEL', 'DATABASE_ID', 'DATABASE_ID_LABEL', 'TIME_FORMAT', 'TIME_FORMAT_LABEL', 'OBS_CONF', 'OBS_CONF_LABEL']
Total Constant: 19


In [7]:
dupe_count = df.duplicated().sum()
print("\nDuplicate Rows:", dupe_count)


Duplicate Rows: 0


In [8]:
missing_count = df.isnull().sum()
missing_count = missing_count[missing_count > 0]
missing_percent = (missing_count / len(df)) * 100
missing_df = pd.DataFrame({"Missing Count": missing_count, "Missing (%)": missing_percent})
print("\nMissing Values:\n", missing_df if not missing_df.empty else "No missing values found.")


Missing Values:
            Missing Count  Missing (%)
OBS_VALUE           2512     1.739395


# Exploratory Data Analysis (EDA)

In [9]:
target_output = 'EDA'
os.makedirs(target_output, exist_ok=True)
meta_output = 'Meta'
os.makedirs(meta_output, exist_ok=True)

In [10]:
output_path = f"{target_output}/missing_values.png"
if FileChecker.is_outdated(output_path, DATASET_PATH):
    missing = df.isnull().sum().sort_values(ascending=False)
    plt.figure(figsize=(12, 6))
    missing.plot(kind="bar")
    plt.title("Missing Values Per Column")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()
    FileChecker.register(output_path, DATASET_PATH)

2026-07-05 21:57:25,948 | FileChecker | INFO | File is up to date, skipping: EDA/missing_values.png


In [11]:
numeric_df = df.select_dtypes(include=np.number)
output_path = f"{target_output}/correlation_heatmap.png"
if FileChecker.is_outdated(output_path, DATASET_PATH):
    plt.figure(figsize=(14, 10))
    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()
    FileChecker.register(output_path, DATASET_PATH)

2026-07-05 21:57:26,093 | FileChecker | INFO | File is up to date, skipping: EDA/correlation_heatmap.png


In [12]:
target_label = 'OBS_VALUE'

output_path = f"{target_output}/target_distribution.png"
if FileChecker.is_outdated(output_path, DATASET_PATH):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[target_label], kde=True)
    plt.title(f"Target Distribution - {target_label}")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()
    FileChecker.register(output_path, DATASET_PATH)

2026-07-05 21:57:26,236 | FileChecker | INFO | File is up to date, skipping: EDA/target_distribution.png


# Data Preprocessing

config

In [13]:
drop_unused = [
    "STRUCTURE", "STRUCTURE_ID", "ACTION", "FREQ",
    "SEX", "AGE",
    "COMP_BREAKDOWN_2", "COMP_BREAKDOWN_3",
    "DATABASE_ID", "TIME_FORMAT", "OBS_CONF",
    "FREQ_LABEL", "REF_AREA_LABEL", "INDICATOR_LABEL",
    "SEX_LABEL", "AGE_LABEL", "URBANISATION_LABEL",
    "UNIT_MEASURE_LABEL", "COMP_BREAKDOWN_1_LABEL",
    "COMP_BREAKDOWN_2_LABEL", "COMP_BREAKDOWN_3_LABEL",
    "AGG_METHOD_LABEL", "DATABASE_ID_LABEL", "UNIT_MULT_LABEL",
    "UNIT_TYPE_LABEL", "TIME_FORMAT_LABEL",
    "OBS_STATUS_LABEL", "OBS_CONF_LABEL",
    "UNIT_MEASURE", "UNIT_TYPE", "AGG_METHOD", "UNIT_MULT"
]
target_fixed     = ["OBS_VALUE", "TIME_PERIOD"]
target_spesific  = ["TIME_PERIOD", "URBANISATION"]
ds_col, dec_col  = "DATE_STAMPS", "DECADE"
target_grouped   = ["REF_AREA", "INDICATOR"]
target_sorted    = ["REF_AREA", "INDICATOR", "TIME_PERIOD"]
nan_threshold    = 0.3
target_encode    = ["REF_AREA", "INDICATOR", "COMP_BREAKDOWN_1"]
output_encode    = ["COUNTRY_ID", "INDICATOR_ID", "COMP_ID"]
scale_col, scale_out = "OBS_VALUE", "OBS_VALUE_SCALED"
scale_groups     = ["COUNTRY_ID", "INDICATOR_ID", "COMP_ID"]
output_path      = "Output/Model"
processed_path   = "Source/Dataset/"
processed_filename = "processed_data_v2.csv"

In [14]:
proc_df = df.copy()
encoder = {}
scale_params = {}

Drop Constant

In [15]:
constant_cols = [c for c in proc_df.columns if proc_df[c].nunique(dropna=False) <= 1]
proc_df.drop(columns=constant_cols, inplace=True)

Drop Unused

In [16]:
existing = [c for c in drop_unused if c in proc_df.columns]
proc_df.drop(columns=existing, inplace=True)

Drop Duplicates

In [17]:
proc_df.drop_duplicates(inplace=True)

Fix Types

In [18]:
for col in target_fixed:
    if col in proc_df.columns:
        proc_df[col] = pd.to_numeric(proc_df[col], errors='coerce')

Spesific Types

In [19]:
proc_df[ds_col] = pd.to_datetime(proc_df[target_spesific[0]].astype(str) + '-01-01')
proc_df[dec_col] = (proc_df[ds_col].dt.year // 10) * 10
proc_df = proc_df[proc_df[target_spesific[1]] == '_T'].copy()
proc_df.drop(columns=[target_spesific[1]], inplace=True)

Column Types Seperation

In [20]:
numerical_cols = proc_df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = proc_df.select_dtypes(exclude=np.number).columns.tolist()
print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

Numerical Columns: ['TIME_PERIOD', 'OBS_VALUE', 'DECADE']
Categorical Columns: ['REF_AREA', 'INDICATOR', 'COMP_BREAKDOWN_1', 'OBS_STATUS', 'DATE_STAMPS']


Handling Missing Values

In [21]:
if proc_df[scale_col].isna().sum() > 0:
    proc_df.sort_values(target_sorted, inplace=True)
    series_nan_pct = proc_df.groupby(target_grouped)[scale_col].apply(lambda x: x.isna().mean())
    bad_series = series_nan_pct[series_nan_pct > nan_threshold].index
    proc_df = proc_df[~proc_df.set_index(target_grouped).index.isin(bad_series)].reset_index(drop=True)
    proc_df[scale_col] = proc_df.groupby(target_grouped)[scale_col].transform(
        lambda x: x.interpolate(method='linear', limit=3, limit_direction='both')
    )
    proc_df.dropna(subset=[scale_col], inplace=True)

Handling Outliers

In [22]:
def mod_z_score(series):
    median = series.median()
    deviation = np.abs(series - median).median()
    z_score = 0.6745 * (series - median) / (deviation + 1e-9)
    return (np.abs(z_score) > 3.5).sum()

check_outliers = proc_df.groupby(target_grouped)[scale_col].apply(mod_z_score).sum()
if check_outliers > 0:
    proc_df[scale_col] = np.log1p(proc_df[scale_col].clip(lower=0))

Encodings

In [23]:
for col, out in zip(target_encode, output_encode):
    if col in proc_df.columns:
        le = LabelEncoder()
        proc_df[out] = le.fit_transform(proc_df[col])
        encoder[col] = le
        proc_df.drop(columns=[col], inplace=True)

In [24]:
if "OBS_STATUS" in proc_df.columns:
    proc_df["OBS_STATUS_ID"] = proc_df["OBS_STATUS"].map({'A': 1, 'O': 0})
    proc_df.drop(columns=["OBS_STATUS"], inplace=True)

Scaling/Normalized

In [25]:
# Preserving Original target values
proc_df["OBS_VALUE_ORIGINAL"] = proc_df[scale_col].copy()

In [26]:
def min_max_scaling(series):
    mn, mx = series.min(), series.max()
    scale_params[series.name] = (mn, mx)
    return (series - mn) / (mx - mn) if mx != mn else pd.Series(0.0, index=series.index)

proc_df[scale_out] = proc_df.groupby(scale_groups)[scale_col].transform(min_max_scaling)

# Save Processed Dataset

In [27]:
os.makedirs(output_path, exist_ok=True)
os.makedirs(processed_path, exist_ok=True)

proc_df.to_csv(os.path.join(processed_path, processed_filename), index=False)
with open(os.path.join(output_path, 'encoders.pkl'), 'wb') as f:
    pickle.dump(encoder, f)
with open(os.path.join(output_path, 'scale_params.pkl'), 'wb') as f:
    pickle.dump(scale_params, f)

processed dataset shapes

In [28]:
print(f"Processed shape: {proc_df.shape}")
proc_df.head()

Processed shape: (121876, 10)


,TIME_PERIOD,OBS_VALUE,DATE_STAMPS,DECADE,COUNTRY_ID,INDICATOR_ID,COMP_ID,OBS_STATUS_ID,OBS_VALUE_ORIGINAL,OBS_VALUE_SCALED
0,2000,4.529368,2000-01-01,2000,0,0,13,1,4.529368,0.0
1,2001,4.615121,2001-01-01,2000,0,0,13,1,4.615121,1.0
2,2002,4.615121,2002-01-01,2000,0,0,13,1,4.615121,1.0
3,2003,4.615121,2003-01-01,2000,0,0,13,1,4.615121,1.0
4,2004,4.615121,2004-01-01,2000,0,0,13,1,4.615121,1.0


# Postprocessing EDA

In [29]:
post_output = "Postprocessing EDA"
os.makedirs(post_output, exist_ok=True)

In [30]:
output_path = f"{post_output}/feature_distributions.png"
if FileChecker.is_outdated(output_path, DATASET_PATH):
    num_cols = proc_df.select_dtypes(include=np.number).columns.tolist()
    ncols = 3
    nrows = int(np.ceil(len(num_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
    axes = np.array(axes).flatten()
    for i, col in enumerate(num_cols):
        sns.histplot(proc_df[col], kde=True, ax=axes[i], color=sns.color_palette("viridis", len(num_cols))[i])
        axes[i].set_title(f"Distribution — {col}", fontsize=10, fontweight="bold")
        axes[i].annotate(f"skew={proc_df[col].skew():.2f}", xy=(0.97, 0.92), xycoords="axes fraction", ha="right", fontsize=8)
    for j in range(len(num_cols), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Post-Preprocessing Feature Distributions", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FileChecker.register(output_path, DATASET_PATH)


In [31]:
output_path = f"{post_output}/feature_boxplots.png"
if FileChecker.is_outdated(output_path, DATASET_PATH):
    num_cols = proc_df.select_dtypes(include=np.number).columns.tolist()
    ncols = 3
    nrows = int(np.ceil(len(num_cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 3.5))
    axes = np.array(axes).flatten()
    palette = sns.color_palette("viridis", len(num_cols))
    for i, col in enumerate(num_cols):
        sns.boxplot(data=proc_df[[col]], y=col, ax=axes[i], color=palette[i])
        q1, q3 = proc_df[col].quantile(0.25), proc_df[col].quantile(0.75)
        iqr = q3 - q1
        n_out = ((proc_df[col] < q1 - 1.5*iqr) | (proc_df[col] > q3 + 1.5*iqr)).sum()
        axes[i].set_title(f"Boxplot — {col}", fontsize=10, fontweight="bold")
        axes[i].annotate(f"IQR outliers: {n_out}", xy=(0.5, 0.02), xycoords="axes fraction", ha="center", fontsize=8, color="crimson")
    for j in range(len(num_cols), len(axes)):
        axes[j].set_visible(False)
    fig.suptitle("Post-Preprocessing Feature Boxplots", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FileChecker.register(output_path, DATASET_PATH)


In [32]:
output_path = f"{post_output}/missing_values_check.png"
if FileChecker.is_outdated(output_path, DATASET_PATH):
    missing = proc_df.isnull().sum().sort_values(ascending=False)
    total = len(proc_df)
    colors = ["crimson" if v > 0 else "mediumseagreen" for v in missing.values]
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(missing.index, missing.values, color=colors)
    for bar, v in zip(bars, missing.values):
        label = f"{v} ({v/total*100:.2f}%)" if v > 0 else "0"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, label, ha="center", fontsize=8)
    ax.set_title("Missing Values Per Column (Post-Preprocessing)\nGreen = None | Red = Residual", fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", labelrotation=30)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FileChecker.register(output_path, DATASET_PATH)

C:\Users\Evan Fanuel\AppData\Local\Temp\ipykernel_16820\3212775926.py:13: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


In [33]:
print(f"Post-preprocessing EDA (outliers, distributions, missing values) saved to: {post_output}")

Post-preprocessing EDA (outliers, distributions, missing values) saved to: Postprocessing EDA


# Data Splitting

In [34]:
test_size = 0.2
val_size  = 0.1

def data_split(data):
    """data: array-like or DataFrame, sequential split (no shuffle, preserves time order)."""
    n = len(data)
    test_n  = max(1, int(n * test_size))
    val_n   = max(1, int(n * val_size))
    train_n = n - test_n - val_n

    train = data[:train_n]
    val   = data[train_n : train_n + val_n]
    test  = data[train_n + val_n:]
    return train, val, test

In [35]:
# New Processed Sorted after Preprocessing
target_sorted_encoded = ["COUNTRY_ID", "INDICATOR_ID", "TIME_PERIOD"]

In [36]:
proc_df_sorted = proc_df.sort_values(target_sorted_encoded).reset_index(drop=True)
train_df, val_df, test_df = data_split(proc_df_sorted)

print(f"Train: {len(train_df)}")
print(f"Test: {len(test_df)}")
print(f"Val: {len(val_df)}")

Train: 85314
Test: 24375
Val: 12187


# Model Selection

config

In [37]:
group_cols   = ['COUNTRY_ID', 'INDICATOR_ID', 'COMP_ID']
period_col   = 'TIME_PERIOD'
target_col   = 'OBS_VALUE'
lag_features = [1, 2, 3]
roll_mean_col, roll_std_col, yoy_col = 'ROLLING_MEAN_3', 'ROLLING_STD_3', 'YOY_DIFF'

Feature Selection & Engineering

In [38]:
# Prophet uses only raw feature selection because the inputs are raw datestamps (ds) and target (y)
prophet_groups = proc_df.groupby(group_cols)
print(f"Prophet -> {prophet_groups.ngroups} series identified")

Prophet -> 4782 series identified


In [39]:
seq_len = 3

def build_gru_sequences(df):
    groups = df.groupby(group_cols)
    all_X, all_Y, all_keys = [], [], []

    for (country, indicator, comp_break), group in groups:
        group = group.sort_values('TIME_PERIOD').reset_index(drop=True)
        series = group['OBS_VALUE_SCALED'].values.astype(np.float32)

        if len(series) <= seq_len:
            continue

        for i in range(len(series) - seq_len):
            all_X.append(series[i: i + seq_len])
            all_Y.append(series[i + seq_len])
            all_keys.append((country, indicator, comp_break))

    X = np.array(all_X, dtype=np.float32).reshape(-1, seq_len, 1)
    Y = np.array(all_Y, dtype=np.float32).reshape(-1, 1)
    return X, Y, all_keys

X_gru, Y_gru, keys_gru = build_gru_sequences(proc_df)
print(f"GRU Sequence Windows: {len(X_gru)}")

GRU Sequence Windows: 107530


In [40]:
def build_catboost_features(df):
    feature_df = df.copy().sort_values(group_cols + [period_col]).reset_index(drop=True)

    for lag in lag_features:
        feature_df[f'LAG_{lag}'] = feature_df.groupby(group_cols)[target_col].shift(lag)

    feature_df[roll_mean_col] = feature_df.groupby(group_cols)[target_col] \
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())

    feature_df[roll_std_col] = feature_df.groupby(group_cols)[target_col] \
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0))

    feature_df[yoy_col] = feature_df.groupby(group_cols)[target_col].diff(1)

    feature_df.dropna(inplace=True)
    feature_df.reset_index(drop=True, inplace=True)
    return feature_df

catboost_required_cols = ['COUNTRY_ID', 'INDICATOR_ID', 'COMP_ID', 'TIME_PERIOD', 'OBS_VALUE', 'DECADE']
catboost_feature_df = build_catboost_features(proc_df)

lag_cols = [f'LAG_{lag}' for lag in lag_features]
catboost_all_features = catboost_required_cols + lag_cols + [roll_mean_col, roll_std_col, yoy_col]

X_cb = catboost_feature_df[catboost_all_features].values
Y_cb = catboost_feature_df[target_col].values
keys_cb = list(zip(
    catboost_feature_df[group_cols[0]],
    catboost_feature_df[group_cols[1]],
    catboost_feature_df[group_cols[2]]
))
print(f"CatBoost Features (After Engineering): {len(catboost_all_features)} | Samples: {len(X_cb)}")

CatBoost Features (After Engineering): 12 | Samples: 107530


# Model Training

Prophet Train

In [41]:
prophet_models, prophet_splits = {}, {}
prophet_true_by_series, prophet_pred_by_series = {}, {}

In [42]:
def fit_one_series(key, group):
    g = group[['DATE_STAMPS', 'OBS_VALUE']].rename(columns={'DATE_STAMPS': 'ds', 'OBS_VALUE': 'y'})
    g['ds'] = pd.to_datetime(g['ds'])
    g = g.sort_values('ds').drop_duplicates(subset='ds').reset_index(drop=True)

    if len(g) < 5:
        return key, None, None, None, None

    train_g, val_g, test_g = data_split(g)
    if len(train_g) < 2:
        return key, None, None, None, None

    cv = train_g['y'].std() / (abs(train_g['y'].mean()) + 1e-9)
    if cv > 1.5:
        naive_pred = np.full(len(test_g), train_g['y'].iloc[-1])
        return key, None, (train_g, val_g, test_g), np.expm1(test_g['y'].values), np.expm1(naive_pred.clip(min=0))

    n_cp = max(1, min(5, len(train_g) // 4))
    m = Prophet(
        seasonality_mode="additive", yearly_seasonality=False,
        weekly_seasonality=False, daily_seasonality=False,
        changepoint_prior_scale=0.5, changepoint_range=0.9,
        n_changepoints=n_cp, interval_width=0.95
    )
    m.fit(train_g)
    return key, m, (train_g, val_g, test_g), None, None

results = Parallel(n_jobs=-1, verbose=5)(
    delayed(fit_one_series)(key, group) for key, group in prophet_groups
)

prophet_models, prophet_splits = {}, {}
prophet_true_by_series, prophet_pred_by_series = {}, {}

for key, m, splits, true_fallback, pred_fallback in results:
    if splits is None:
        continue
    if m is not None:
        prophet_models[key] = m
        prophet_splits[key] = splits
    else:
        prophet_true_by_series[key] = true_fallback
        prophet_pred_by_series[key] = pred_fallback

print(f"Prophet: fitted {len(prophet_models)} per-series models")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:    6.3s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:    8.0s
[Parallel(n_jobs=-1)]: Done 256 tasks      | elapsed:   10.4s
[Parallel(n_jobs=-1)]: Done 418 tasks      | elapsed:   13.3s
[Parallel(n_jobs=-1)]: Done 616 tasks      | elapsed:   16.7s
[Parallel(n_jobs=-1)]: Done 850 tasks      | elapsed:   20.9s
[Parallel(n_jobs=-1)]: Done 1120 tasks      | elapsed:   25.7s
[Parallel(n_jobs=-1)]: Done 1426 tasks      | elapsed:   31.1s
[Parallel(n_jobs=-1)]: Done 1768 tasks      | elapsed:   37.1s
[Parallel(n_jobs=-1)]: Done 2146 tasks      | elapsed:   44.4s
[Parallel(n_jobs=-1)]: Done 2560 tasks      | elapsed:   52.0s
[Parallel(n_jobs=-1)]: Done 3010 tasks      | elapsed:   59.1s
[Parallel(n_jobs=-1)]: Done 3496 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 4018 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 4576 tasks      | e

Prophet: fitted 3954 per-series models


[Parallel(n_jobs=-1)]: Done 4782 out of 4782 | elapsed:  1.5min finished


GRU Train

In [43]:
Xg_train, Xg_val, Xg_test = data_split(X_gru)
Yg_train, Yg_val, Yg_test = data_split(Y_gru)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [44]:
class GRUNet(TorchModule):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2, dropout=0.3):
        super().__init__()
        self.gru = GRU_setup(input_size, hidden_size, num_layers,
                              batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = linear_method(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

gru_model = GRUNet().to(device)
optimizer = model_optimizer(gru_model.parameters(), lr=0.001)
criterion = loss_function()

In [45]:
Xg_train_t = torch.tensor(Xg_train).to(device)
Yg_train_t = torch.tensor(Yg_train).to(device)
Xg_val_t   = torch.tensor(Xg_val).to(device)
Yg_val_t   = torch.tensor(Yg_val).to(device)

In [46]:
epochs, batch_size = 50, 32
for epoch in range(epochs):
    gru_model.train()
    perm = torch.randperm(len(Xg_train_t))
    for i in range(0, len(perm), batch_size):
        idx = perm[i:i + batch_size]
        optimizer.zero_grad()
        pred = gru_model(Xg_train_t[idx])
        loss = criterion(pred, Yg_train_t[idx])
        loss.backward()
        optimizer.step()

    gru_model.eval()
    with torch.no_grad():
        val_loss = criterion(gru_model(Xg_val_t), Yg_val_t)
    if epoch % 10 == 0:
        print(f"[GRU] Epoch {epoch} | Val Loss: {val_loss.item():.5f}")

[GRU] Epoch 0 | Val Loss: 0.02567
[GRU] Epoch 10 | Val Loss: 0.02418
[GRU] Epoch 20 | Val Loss: 0.02408
[GRU] Epoch 30 | Val Loss: 0.02450
[GRU] Epoch 40 | Val Loss: 0.02423


Catboostregressor Train

In [47]:
Xc_train, Xc_val, Xc_test = data_split(X_cb)
Yc_train, Yc_val, Yc_test = data_split(Y_cb)

In [48]:
catboost_model = CatBoostRegressor(
    iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
    random_seed=42, eval_metric="RMSE", verbose=100, early_stopping_rounds=50
)
catboost_model.fit(Xc_train, Yc_train, eval_set=(Xc_val, Yc_val), use_best_model=True)

0:	learn: 2.0802097	test: 2.0653547	best: 2.0653547 (0)	total: 167ms	remaining: 1m 23s
100:	learn: 0.0492240	test: 0.0631039	best: 0.0584452 (93)	total: 895ms	remaining: 3.54s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.05844516896
bestIteration = 93

Shrink model to first 94 iterations.


CatBoostRegressor(depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=500, l2_leaf_reg=3.0, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

# Evaluation

In [49]:
def safe_mape(true, pred):
    mask = true > 0
    if mask.sum() == 0:
        return float('nan')
    return mean_absolute_percentage_error(true[mask], pred[mask]) * 100

In [50]:
def median_metrics(true_by_series, pred_by_series, min_test_for_r2=6):
    per_mae, per_mape, per_rmse, per_r2 = [], [], [], []
    for key in true_by_series:
        t, p = np.array(true_by_series[key]), np.array(pred_by_series[key])
        per_mae.append(mean_absolute_error(t, p))
        per_mape.append(safe_mape(t, p))
        per_rmse.append(root_mean_squared_error(t, p))
        if len(t) >= min_test_for_r2:
            per_r2.append(r2_score(t, p))
    return {
        "Median_MAE":  round(float(np.median(per_mae)), 4),
        "Median_RMSE": round(float(np.median(per_rmse)), 4),
        "Median_MAPE": round(float(np.median([m for m in per_mape if not np.isnan(m)])), 4),
        "Median_R2":   round(float(np.median(per_r2)), 4) if per_r2 else 'N/A',
        "R2_series_count": len(per_r2)
    }

Prophet Evaluation

In [51]:
def predict_one_series(key, m, splits):
    _, _, test_g = splits
    if test_g.empty:
        return key, None, None
    forecast = m.predict(test_g[['ds']])
    true_vals = np.expm1(test_g['y'].values)
    pred_vals = np.expm1(forecast['yhat'].values.clip(min=0))
    return key, true_vals, pred_vals

eval_results = Parallel(n_jobs=-1, verbose=5)(
    delayed(predict_one_series)(key, m, prophet_splits[key])
    for key, m in prophet_models.items()
)

for key, true_vals, pred_vals in eval_results:
    if true_vals is None:
        continue
    prophet_true_by_series[key] = true_vals
    prophet_pred_by_series[key] = pred_vals

prophet_metrics = median_metrics(prophet_true_by_series, prophet_pred_by_series)
print("Prophet:", prophet_metrics)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done 148 tasks      | elapsed:    5.9s
[Parallel(n_jobs=-1)]: Done 624 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done 1272 tasks      | elapsed:   11.2s
[Parallel(n_jobs=-1)]: Done 2064 tasks      | elapsed:   14.6s
[Parallel(n_jobs=-1)]: Done 3000 tasks      | elapsed:   18.6s
[Parallel(n_jobs=-1)]: Done 3954 out of 3954 | elapsed:   22.6s finished


Prophet: {'Median_MAE': 0.627, 'Median_RMSE': 0.7253, 'Median_MAPE': 26.4372, 'Median_R2': -3.5129, 'R2_series_count': 1321}


GRU Evaluation

In [52]:
gru_model.eval()
with torch.no_grad():
    gru_pred_scaled = gru_model(torch.tensor(Xg_test).to(device)).cpu().numpy().flatten()
gru_true_scaled = Yg_test.flatten()
_, _, test_keys_gru = data_split(keys_gru)

gru_true_by_series, gru_pred_by_series = defaultdict(list), defaultdict(list)
for i, key in enumerate(test_keys_gru):
    mn, mx = scale_params.get(key, (0, 1))
    actual    = gru_true_scaled[i] * (mx - mn) + mn
    predicted = gru_pred_scaled[i] * (mx - mn) + mn
    gru_true_by_series[key].append(np.expm1(max(actual, 0)))
    gru_pred_by_series[key].append(np.expm1(max(predicted, 0)))

gru_metrics = median_metrics(gru_true_by_series, gru_pred_by_series)
print("GRU:", gru_metrics)

GRU: {'Median_MAE': 0.2528, 'Median_RMSE': 0.3886, 'Median_MAPE': 11.9933, 'Median_R2': 0.895, 'R2_series_count': 965}


Catboostregressor evaluation

In [53]:
cb_pred = catboost_model.predict(Xc_test)
cb_true = Yc_test
_, _, test_keys_cb = data_split(keys_cb)

# ── Build per-series dicts (same convention as Prophet/GRU) ─────────────────
cb_true_by_series, cb_pred_by_series = defaultdict(list), defaultdict(list)
for i, key in enumerate(test_keys_cb):
    cb_true_by_series[key].append(cb_true[i])
    cb_pred_by_series[key].append(cb_pred[i])

# ── Use the shared median_metrics function (consistent with all models) ────
catboost_metrics = median_metrics(cb_true_by_series, cb_pred_by_series)
print("CatBoost:", catboost_metrics)

CatBoost: {'Median_MAE': 0.0258, 'Median_RMSE': 0.0307, 'Median_MAPE': 2.2735, 'Median_R2': 0.8864, 'R2_series_count': 965}


Stratified Metrics

In [54]:
def stratified_metrics(true_by_series, pred_by_series, model_name="Model"):
    thresholds = [1, 4, 6, 10, 15, 20]
    rows = []

    for min_len in thresholds:
        per_mae, per_mape, per_rmse, per_r2 = [], [], [], []

        for key in true_by_series:
            t, p = np.array(true_by_series[key]), np.array(pred_by_series[key])
            if len(t) < min_len:
                continue

            per_mae.append(mean_absolute_error(t, p))
            per_mape.append(safe_mape(t, p))
            per_rmse.append(root_mean_squared_error(t, p))
            if len(t) > 1:
                per_r2.append(r2_score(t, p))

        rows.append({
            "Model": model_name,
            "Min_Test_Len": min_len,
            "Series_Count": len(per_mae),
            "Median_MAE":  round(float(np.median(per_mae)), 4) if per_mae else None,
            "Median_RMSE": round(float(np.median(per_rmse)), 4) if per_rmse else None,
            "Median_MAPE": round(float(np.median([m for m in per_mape if not np.isnan(m)])), 4) if per_mape else None,
            "Median_R2":   round(float(np.median(per_r2)), 4) if per_r2 else None,
        })

    return pd.DataFrame(rows)


In [55]:
prophet_strat  = stratified_metrics(prophet_true_by_series, prophet_pred_by_series, "Prophet")
print(prophet_strat.to_string(index=False))
gru_strat      = stratified_metrics(gru_true_by_series, gru_pred_by_series, "GRU")
print(gru_strat.to_string(index=False))
catboost_strat = stratified_metrics(cb_true_by_series, cb_pred_by_series, "CatBoost")  # or whatever you named CatBoost's per-series dicts
print(catboost_strat.to_string(index=False))

  Model  Min_Test_Len  Series_Count  Median_MAE  Median_RMSE  Median_MAPE  Median_R2
Prophet             1          4782      0.6270       0.7253      26.4372    -0.3333
Prophet             4          4782      0.6270       0.7253      26.4372    -0.3333
Prophet             6          1321      2.0884       2.3904      20.2557    -3.5129
Prophet            10             0         NaN          NaN          NaN        NaN
Prophet            15             0         NaN          NaN          NaN        NaN
Prophet            20             0         NaN          NaN          NaN        NaN
Model  Min_Test_Len  Series_Count  Median_MAE  Median_RMSE  Median_MAPE  Median_R2
  GRU             1           966      0.2528       0.3886      11.9933      0.895
  GRU             4           965      0.2549       0.3894      12.0133      0.895
  GRU             6           965      0.2549       0.3894      12.0133      0.895
  GRU            10           965      0.2549       0.3894      12.0133  

# Single-series evaluation

In [56]:
target_comp_id = list(encoder['COMP_BREAKDOWN_1'].classes_).index('WB_SE4ALL_TYPE_MDRNRENWBLS')
print(target_comp_id)

10


Config Series

In [57]:
target_country     = 'UGA'
target_indicator   = 'WB_SE4ALL_EG_FCON_RNEW'
target_compbreak   = 'WB_SE4ALL_TYPE_HEATRSNG'

target_country_id   = encoder['REF_AREA'].transform([target_country])[0]
target_indicator_id = encoder['INDICATOR'].transform([target_indicator])[0]
target_comp_id       = encoder['COMP_BREAKDOWN_1'].transform([target_compbreak])[0]

print(f"COUNTRY_ID={target_country_id}, INDICATOR_ID={target_indicator_id}, COMP_ID={target_comp_id}")

series_df = proc_df[
    (proc_df['COUNTRY_ID'] == target_country_id) &
    (proc_df['INDICATOR_ID'] == target_indicator_id) &
    (proc_df['COMP_ID'] == target_comp_id)
].sort_values('TIME_PERIOD').reset_index(drop=True)

print(f"Series length: {len(series_df)}")
print(series_df[['TIME_PERIOD', 'OBS_VALUE', 'OBS_VALUE_ORIGINAL', 'OBS_VALUE_SCALED']])

COUNTRY_ID=221, INDICATOR_ID=3, COMP_ID=9
Series length: 32
    TIME_PERIOD  OBS_VALUE  OBS_VALUE_ORIGINAL  OBS_VALUE_SCALED
0          1990   5.697429            5.697429          0.000000
1          1991   5.718671            5.718671          0.022084
2          1992   5.745564            5.745564          0.050043
3          1993   5.762680            5.762680          0.067838
4          1994   5.784133            5.784133          0.090141
5          1995   5.793014            5.793014          0.099374
6          1996   5.808142            5.808142          0.115103
7          1997   5.762051            5.762051          0.067184
8          1998   5.791488            5.791488          0.097788
9          1999   5.816516            5.816516          0.123808
10         2000   5.860216            5.860216          0.169241
11         2001   5.947251            5.947251          0.259727
12         2002   5.982676            5.982676          0.296556
13         2003   6.017863    

Prophet single-series

In [58]:
g = series_df[['DATE_STAMPS', 'OBS_VALUE']].rename(columns={'DATE_STAMPS': 'ds', 'OBS_VALUE': 'y'})
g['ds'] = pd.to_datetime(g['ds'])
g = g.sort_values('ds').drop_duplicates(subset='ds').reset_index(drop=True)

train_g, val_g, test_g = data_split(g)

m_single = Prophet(
    seasonality_mode="additive", yearly_seasonality=False,
    weekly_seasonality=False, daily_seasonality=False,
    changepoint_prior_scale=0.05, changepoint_range=1,
    n_changepoints=5, interval_width=0.95
)
m_single.fit(train_g)
forecast_single = m_single.predict(test_g[['ds']])

prophet_true_single = np.expm1(test_g['y'].values)
prophet_pred_single = np.expm1(forecast_single['yhat'].values.clip(min=0))

mae_p  = mean_absolute_error(prophet_true_single, prophet_pred_single)
rmse_p = root_mean_squared_error(prophet_true_single, prophet_pred_single)
mape_p = safe_mape(prophet_true_single, prophet_pred_single)
r2_p   = r2_score(prophet_true_single, prophet_pred_single) if len(prophet_true_single) > 1 else float('nan')

print(f"\nProphet (single series) -> MAE: {mae_p:.4f}, RMSE: {rmse_p:.4f}, MAPE: {mape_p:.4f}%, R2: {r2_p:.4f}")

22:07:53 - cmdstanpy - INFO - Chain [1] start processing
22:07:54 - cmdstanpy - INFO - Chain [1] done processing



Prophet (single series) -> MAE: 17.1973, RMSE: 20.7310, MAPE: 2.4123%, R2: 0.8002


GRU Single-Series

In [59]:
single_seq = series_df['OBS_VALUE_SCALED'].values.astype(np.float32)
Xg_single, Yg_single = [], []
for i in range(len(single_seq) - seq_len):
    Xg_single.append(single_seq[i:i + seq_len])
    Yg_single.append(single_seq[i + seq_len])
Xg_single = np.array(Xg_single, dtype=np.float32).reshape(-1, seq_len, 1)
Yg_single = np.array(Yg_single, dtype=np.float32).reshape(-1, 1)

_, _, Xg_single_test = data_split(Xg_single)
_, _, Yg_single_test = data_split(Yg_single)

gru_model.eval()
with torch.no_grad():
    pred_scaled = gru_model(torch.tensor(Xg_single_test).to(device)).cpu().numpy().flatten()
true_scaled = Yg_single_test.flatten()

mn, mx = scale_params.get((target_country_id, target_indicator_id, target_comp_id), (0, 1))
gru_true_single = np.expm1(np.clip(true_scaled * (mx - mn) + mn, 0, None))
gru_pred_single = np.expm1(np.clip(pred_scaled * (mx - mn) + mn, 0, None))

mae_g  = mean_absolute_error(gru_true_single, gru_pred_single)
rmse_g = root_mean_squared_error(gru_true_single, gru_pred_single)
mape_g = safe_mape(gru_true_single, gru_pred_single)
r2_g   = r2_score(gru_true_single, gru_pred_single) if len(gru_true_single) > 1 else float('nan')

print(f"GRU (single series) -> MAE: {mae_g:.4f}, RMSE: {rmse_g:.4f}, MAPE: {mape_g:.4f}%, R2: {r2_g:.4f}")

GRU (single series) -> MAE: 28.9779, RMSE: 35.6463, MAPE: 3.8941%, R2: 0.3452


Catboostregressor Single-Series

In [60]:
single_features = build_catboost_features(series_df)
X_single = single_features[catboost_all_features].values
Y_single = single_features[target_col].values

_, _, Xc_single_test = data_split(X_single)
_, _, Yc_single_test = data_split(Y_single)

cb_pred_single = catboost_model.predict(Xc_single_test)
cb_true_single = Yc_single_test

cb_true_single = np.expm1(cb_true_single)
cb_pred_single = np.expm1(cb_pred_single)  # clip only if needed for stability

mae_c  = mean_absolute_error(cb_true_single, cb_pred_single)
rmse_c = root_mean_squared_error(cb_true_single, cb_pred_single)
mape_c = safe_mape(cb_true_single, cb_pred_single)
r2_c   = r2_score(cb_true_single, cb_pred_single) if len(cb_true_single) > 1 else float('nan')

print(f"CatBoost (single series) -> MAE: {mae_c:.4f}, RMSE: {rmse_c:.4f}, MAPE: {mape_c:.4f}%, R2: {r2_c:.4f}")

CatBoost (single series) -> MAE: 40.0259, RMSE: 56.5112, MAPE: 5.2156%, R2: -0.6457


Evaluation Table

In [61]:
prophet_single_metrics = {
    "MAE": round(mae_p, 4), "RMSE": round(rmse_p, 4),
    "MAPE (%)": round(mape_p, 4), "R2": round(r2_p, 4)
}

# Save Model

In [62]:
model_output_dir = "Output/models"
os.makedirs(model_output_dir, exist_ok=True)

Prophet

In [63]:
with open(f"{model_output_dir}/prophet_models.pkl", "wb") as f:
    pickle.dump(prophet_models, f)

# Single Series Prophet
if 'm_single' in dir():
    joblib.dump(m_single, f"{model_output_dir}/prophet_single_series.pkl")

GRU

In [64]:
torch.save(gru_model.state_dict(), f"{model_output_dir}/gru_model.pt")

Catboost

In [65]:
catboost_model.save_model(f"{model_output_dir}/catboost_model.cbm")

# Model Comparison 

In [66]:
evaluation_comparison_table = pd.DataFrame({
    "Model": ["Prophet", "GRU", "CatBoost"],
    "Scope": ["Single series (Uganda—Renewable Energy Consumption—Heat and Cooling)",
              "Pooled (median across series)", "Pooled (median across series)"],
    "MAE": [prophet_single_metrics["MAE"], gru_metrics["Median_MAE"], catboost_metrics["Median_MAE"]],
    "RMSE": [prophet_single_metrics["RMSE"], gru_metrics["Median_RMSE"], catboost_metrics["Median_RMSE"]],
    "MAPE (%)": [prophet_single_metrics["MAPE (%)"], gru_metrics["Median_MAPE"], catboost_metrics["Median_MAPE"]],
    "R2": [prophet_single_metrics["R2"], gru_metrics["Median_R2"], catboost_metrics["Median_R2"]],
})
print("\n", evaluation_comparison_table)


       Model                                              Scope      MAE  \
0   Prophet  Single series (Uganda—Renewable Energy Consump...  17.1973   
1       GRU                      Pooled (median across series)   0.2528   
2  CatBoost                      Pooled (median across series)   0.0258   

      RMSE  MAPE (%)      R2  
0  20.7310    2.4123  0.8002  
1   0.3886   11.9933  0.8950  
2   0.0307    2.2735  0.8864  


In [67]:
comparison_output = "Output/model_evaluation_comparison"
os.makedirs(comparison_output, exist_ok=True)
comparison_path = f"{comparison_output}/model_comparison_final.png"

if FileChecker.is_outdated(comparison_path, DATASET_PATH):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Model Comparison — Actual vs Predicted", fontsize=13, fontweight="bold")

    # Prophet: single-series scatter
    ax = axes[0]
    ax.scatter(prophet_true_single, prophet_pred_single, alpha=0.7, s=40, color="steelblue")
    lim = max(prophet_true_single.max(), prophet_pred_single.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", linewidth=1.5)
    info = "\n".join([f"{k}: {v}" for k, v in prophet_single_metrics.items()])
    ax.text(0.03, 0.97, info, transform=ax.transAxes, fontsize=8, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.4))
    ax.set_title("Prophet (single series)")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(True, alpha=0.3)

    # GRU: pooled scatter 
    ax = axes[1]
    ax.scatter(gru_true_scaled, gru_pred_scaled, alpha=0.3, s=10, color="seagreen")
    lim = max(gru_true_scaled.max(), gru_pred_scaled.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", linewidth=1.5)
    info = "\n".join([f"{k}: {v}" for k, v in gru_metrics.items()])
    ax.text(0.03, 0.97, info, transform=ax.transAxes, fontsize=7, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.4))
    ax.set_title("GRU (pooled, all series)")
    ax.set_xlabel("Actual (scaled)")
    ax.set_ylabel("Predicted (scaled)")
    ax.grid(True, alpha=0.3)

    # CatBoost: pooled scatter
    ax = axes[2]
    ax.scatter(cb_true, cb_pred, alpha=0.3, s=10, color="darkorange")
    lim = max(cb_true.max(), cb_pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", linewidth=1.5)
    info = "\n".join([f"{k}: {v}" for k, v in catboost_metrics.items()])
    ax.text(0.03, 0.97, info, transform=ax.transAxes, fontsize=7, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.4))
    ax.set_title("CatBoost (pooled, all series)")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(comparison_path, dpi=150)
    plt.close()
    FileChecker.register(comparison_path, DATASET_PATH)

print(f"Saved: {comparison_path}")

2026-07-05 22:07:57,908 | FileChecker | INFO | File is up to date, skipping: Output/model_evaluation_comparison/model_comparison_final.png


Saved: Output/model_evaluation_comparison/model_comparison_final.png
